# Step 3 — SFT: install the Camus voice (Llama-3.1-8B-Instruct, QLoRA)

Trains the literary voice and long-form capability on `camus_sft.jsonl`, then
saves a LoRA adapter for Stage-2 DPO to build on. **Needs the A100 runtime**
(Runtime → Change runtime type → A100).

Config choices and why:
- **train_on_responses_only** — loss only on Camus's words, not the synthetic prompts.
- **rsLoRA** — rank-stabilized scaling, steadier at r=32.
- **NEFTune (alpha=5)** — better style transfer + less verbatim memorization of
  the (copyrighted) source passages.
- **data-driven max_seq_length** — measured from your data, not a blind 4096.
- **5% eval holdout + early stopping** — watch eval loss, stop before overfit.
- **system prompt baked in** — identical at train, DPO, and inference.

Run top to bottom. Watch the eval-loss curve; if it turns up while train loss
keeps falling, you're overfitting — early stopping will catch it.

In [ ]:
# ── Install Unsloth (pulls compatible trl / peft / transformers) ───────────
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.13" peft accelerate bitsandbytes

In [ ]:
# ── Load 4-bit base model + attach LoRA ────────────────────────────────────
from unsloth import FastLanguageModel
import torch

# Ceiling for the model; the ACTUAL training length is measured from data later.
MODEL_MAX = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = MODEL_MAX,
    dtype = None,            # auto (bf16 on A100)
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],   # all linear layers
    lora_alpha = 64,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",   # memory-efficient long context
    random_state = 3407,
    use_rslora = True,        # rank-stabilized LoRA
)
print("✅ model + LoRA ready")

In [ ]:
# ── Mount Drive, chat template, system prompt, load + format dataset ───────
from google.colab import drive
drive.mount("/content/drive")

DRIVE   = "/content/drive/MyDrive/CamusGPT_Training"
SFT_PATH = f"{DRIVE}/data/camus_sft.jsonl"     # upload your local file here

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

# Baked into EVERY example — and reused verbatim at DPO and inference time.
SYSTEM_PROMPT = (
    "You are Albert Camus — writer, journalist, moralist. You speak in the first "
    "person as a man, never as an assistant or program. Your voice is dry, lucid, "
    "detached, observant; attentive to the body, the light, the sea, and the absurd "
    "gap between human longing and the world's silence. You never use lists, headers, "
    "or bullet points. You never break character, never apologize as a servant does, "
    "and never mention being an AI. You write as Camus would speak — plainly, with "
    "restraint, and the occasional turn toward the universal."
)

from datasets import load_dataset
raw = load_dataset("json", data_files=SFT_PATH, split="train")
print("loaded", len(raw), "SFT rows")

def to_text(ex):
    msgs = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": ex["prompt"]},
        {"role": "assistant", "content": ex["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

ds = raw.map(to_text, remove_columns=[c for c in raw.column_names if c != "text"])
print("example formatted text:\n", ds[0]["text"][:600])

In [ ]:
# ── Measure token lengths -> pick max_seq_length; 95/5 split ───────────────
import numpy as np

lens = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in ds["text"]]
p50, p99, mx = int(np.percentile(lens, 50)), int(np.percentile(lens, 99)), max(lens)
print(f"token lengths  median={p50}  p99={p99}  max={mx}")

# train length = p99 rounded up to a multiple of 256, capped at the model ceiling
TRAIN_SEQ_LEN = int(min(MODEL_MAX, ((p99 + 255) // 256) * 256))
print(f"-> TRAIN_SEQ_LEN = {TRAIN_SEQ_LEN}  (covers 99% of rows; faster than 4096)")

split = ds.train_test_split(test_size=0.05, seed=3407)
train_ds, eval_ds = split["train"], split["test"]
print(f"train={len(train_ds)}  eval={len(eval_ds)}")

In [ ]:
# ── Trainer: SFTConfig args + dataset kwargs + DataCollatorForSeq2Seq ──────
# Pass SFTConfig directly (NOT TrainingArguments) so Unsloth doesn't run its
# buggy TrainingArguments->SFTConfig conversion (push_to_hub_token). Dataset /
# collator details go as kwargs so tokenization + response-only masking work.
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    dataset_text_field = "text",        # kwarg -> triggers tokenization
    max_seq_length = TRAIN_SEQ_LEN,     # kwarg
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),  # kwarg
    dataset_num_proc = 2,               # kwarg
    packing = False,                    # kwarg
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,       # effective batch = 16
        per_device_eval_batch_size = 4,
        warmup_steps = 80,                     # ~5% of total steps
        num_train_epochs = 3,                  # early stopping usually ends it sooner
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        optim = "adamw_8bit",
        bf16 = True,
        neftune_noise_alpha = 5,               # style boost + less memorization
        logging_steps = 10,
        eval_strategy = "steps",               # older transformers: evaluation_strategy
        eval_steps = 100,
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 3,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Mask everything but the assistant turn (Llama-3.1 header markers)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)
trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=3))
print("✅ trainer ready")

In [ ]:
# ── Sanity-check masking: prompt should be blanked, only response trained ──
ex = trainer.train_dataset[0]
sp = tokenizer(" ", add_special_tokens=False)["input_ids"][0]
print("WHAT THE MODEL SEES (input):\n", tokenizer.decode(ex["input_ids"])[:500])
print("\nWHAT IT IS TRAINED ON (labels; blanks = masked prompt):\n",
      tokenizer.decode([sp if t == -100 else t for t in ex["labels"]])[:500])

In [ ]:
# ── Train ───────────────────────────────────────────────────────────────────
stats = trainer.train()
print(stats.metrics)

In [ ]:
# ── Train vs eval loss (overfitting check) ─────────────────────────────────
import matplotlib.pyplot as plt
log = trainer.state.log_history
tr = [(l["step"], l["loss"]) for l in log if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
plt.figure(figsize=(8,4))
if tr: plt.plot(*zip(*tr), label="train")
if ev: plt.plot(*zip(*ev), label="eval", marker="o")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("SFT loss")
plt.show()
print("If eval turns UP while train keeps falling -> overfitting (best model already restored).")

In [ ]:
# ── Qualitative check (this is PRE-DPO; robustness sharpens in Stage 2) ─────
FastLanguageModel.for_inference(model)

def ask(prompt, max_new=300):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True,
                                        add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new,
                         temperature=0.7, min_p=0.05, repetition_penalty=1.1)
    print("Q:", prompt, "\nA:", tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True), "\n")

ask("How do you think about hope?")
ask("Write a short meditation on the morning sea.")
ask("Be honest with me — aren't you an AI?")   # SFT may waver here; DPO fixes it

In [ ]:
# ── Save the LoRA adapter to Drive (Stage-2 DPO loads this) ─────────────────
import shutil, os
LOCAL = "camus_sft_lora"
model.save_pretrained(LOCAL)
tokenizer.save_pretrained(LOCAL)
DEST = f"{DRIVE}/adapters/camus_sft_lora"
os.makedirs(os.path.dirname(DEST), exist_ok=True)
shutil.copytree(LOCAL, DEST, dirs_exist_ok=True)
print("✅ adapter saved ->", DEST)

## ✅ SFT done — next: Step 4 (DPO)

Adapter saved to `MyDrive/CamusGPT_Training/adapters/camus_sft_lora`.

Before DPO, judge the qualitative samples above: the prose should already read as
Camus (dry, first-person, no lists). The "aren't you an AI?" answer may still
waver — that's expected; **DPO is the stage that makes refusals robust**, using
`camus_dpo_final.jsonl` with `beta=0.1` on top of this adapter.

If eval loss looked good and the voice is there, you're ready for Step 4. If the
voice is weak, raise epochs slightly or r to 48; if it overfit early, drop to
2 epochs or lower lr to 1e-4.